# 04. Path-distance classification

Status: preprocessing and multivariate dynamic time warping variants are explicit. Corrected configurations await execution. Signature features remain open.


## What to run

Needs BasicMotions under `data/raw/BasicMotions/`. Three preprocessing conditions are separate runs, reported separately:

```bash
python scripts/run_classification.py --config configs/classification_basicmotions.yaml
python scripts/run_classification.py --config configs/classification_basicmotions_training_channel.yaml
python scripts/run_classification.py --config configs/classification_basicmotions_per_series.yaml
```

Raw archive run is the anchor: its DTW score reproduces the published baseline, which is what makes the pipeline checkable. Standardised variants change the distance and so cannot serve that role. Seconds each, NumPy and aeon only, no GPU.


## 1. Purpose

Classification tests whether a path discrepancy preserves class-relevant shape without involving a trained reconstruction model. Model, optimiser, and training loss are absent, so performance changes can be attributed to representation and distance. Official train and test splits remain fixed.


## 2. Data and preprocessing

BasicMotions contains 40 training and 40 test paths. Each path $x^{(i)}\in\mathbb R^{6\times100}$ contains three accelerometer and three gyroscope channels sampled at a uniform cadence. Labels are standing, running, walking, and badminton.

Preprocessing is an experimental condition. Raw data use $x^{(i)}$ as distributed. Training-channel standardisation calculates

$$
\mu_c=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}x^{(i)}_{c,r},\qquad
\sigma_c^2=\frac{1}{n_{\mathrm{train}}T}\sum_{i=1}^{n_{\mathrm{train}}}\sum_{r=1}^{T}(x^{(i)}_{c,r}-\mu_c)^2.
$$

Both splits then use $\widetilde x^{(i)}_{c,r}=(x^{(i)}_{c,r}-\mu_c)/\sigma_c$. This changes multivariate distance by weighting channel $c$ by $\sigma_c^{-2}$. Per-series standardisation instead uses one mean and scale for each pair $(i,c)$, calculated over time within that series. Raw, training-channel and per-series results are reported separately. Test labels affect none of these transformations.


## 3. Fixed 1-nearest-neighbour algorithm

For test path $z$, predict label of training path

$$
i^*(z)=\operatorname*{arg\,min}_{1\leq i\leq n_{\mathrm{train}}}d(z,x^{(i)}).
$$

Euclidean distance flattens channel and time coordinates:

$$
d_{\mathrm E}(x,z)=\left(\sum_{c=1}^{6}\sum_{r=1}^{100}|x_{c,r}-z_{c,r}|^2\right)^{1/2}.
$$

Dimension-dependent dynamic time warping uses local multichannel cost $\delta(r,s)=\|x_{:,r}-z_{:,s}\|_2^2$ and recursion

$$
D_{r,s}=\delta(r,s)+\min\{D_{r-1,s},D_{r,s-1},D_{r-1,s-1}\}.
$$

Dimension-independent dynamic time warping aligns each channel separately and adds channel costs:

$$
d_{\mathrm{DTW-I}}(x,z)=\sum_{c=1}^{6}d_{\mathrm{DTW}}(x_c,z_c).
$$

Accuracy is fraction of correct test labels. Balanced accuracy is mean class recall.


In [ ]:
import json
from pathlib import Path

result_paths = {
    name: Path(f'../results/runs/classification_basicmotions_{name}/results.json')
    for name in ('raw', 'training_channel', 'per_series')
}
results = {
    name: json.loads(path.read_text())
    for name, path in result_paths.items() if path.exists()
}
{name: [(row['distance'], row['scores']) for row in result['results']]
 for name, result in results.items()}


## 4. Results

Diagnostic reproduction on 19 August isolated preprocessing before the corrected runner was written:

| preprocessing | Euclidean | DTW-dependent |
|---|---:|---:|
| raw archive | 0.600 | 0.975 |
| training-channel | 0.575 | 0.900 |
| per-series | 0.725 | 0.975 |

Raw DTW-dependent reproduces the published 0.975 result. Training-channel standardisation reweights sensors and changes three of forty DTW predictions. Its 0.900 result describes a different geometry and cannot anchor the archive implementation. Corrected configurations add DTW-independent explicitly and require fresh stored outputs before signature comparison.


## 5. Signature extension

After corrected controls are stored, next representation is a truncated signature or log signature of an explicitly augmented path. Fix augmentation, interpolation, truncation level, input scaling and feature scaling using training information. Apply the same 1-nearest-neighbour rule so representation is the only changed component.

Raw signature uses increments and omits absolute level. Initial study therefore includes a base point or initial value. Time augmentation records cadence; lead-lag augmentation is a later alternative when interaction with increments is the intended feature.


## 6. Reproducibility map

- Data: `data/raw/BasicMotions/`
- Raw configuration: `configs/classification_basicmotions.yaml`
- Training-channel configuration: `configs/classification_basicmotions_training_channel.yaml`
- Per-series configuration: `configs/classification_basicmotions_per_series.yaml`
- Classification functions: `src/pathloss/classification.py`
- Runner: `scripts/run_classification.py`
- Corrected outputs: `results/runs/classification_basicmotions_{raw,training_channel,per_series}/results.json` after execution
- Historical training-channel output: `results/runs/classification_basicmotions/results.json`
